# VRAM Wall — Kaggle runner

Phase-2 spike: APIA (fwd-only + local RLS) vs LoRA (fwd+bwd+AdamW) peak-VRAM divergence.

**Setup:** enable GPU accelerator (Settings → Accelerator → T4/P100). Default config targets
`facebook/opt-1.3b` on a ~16GB card — batch sweep finds where fwd+bwd hits *that* wall.

Outputs land in `/kaggle/working/` for download. Repo: https://github.com/GhostPookie/contextual-consolidation-rc

In [ ]:
# Clone repo + deps (torch/transformers preinstalled on Kaggle)
!git clone -q https://github.com/GhostPookie/contextual-consolidation-rc repo
%cd repo
!pip install -q peft accelerate
import torch
print(torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory/1e9, 'GB')

In [ ]:
# --- CONFIG -----------------------------------------------------------
MODEL   = 'facebook/opt-1.3b'      # or openai-community/gpt2-large
BATCHES = '1,4,8,16,32,64'         # probe upward until LoRA OOM
SEQ     = 128
TARGETS = 'q_proj,v_proj'          # OPT names; GPT-2: c_attn,c_proj
OUT     = '/kaggle/working/vram_opt13b.json'
# ----------------------------------------------------------------------
!python vram_spike.py --model {MODEL} --batches {BATCHES} --seq {SEQ} --targets {TARGETS} --out {OUT}

In [ ]:
import json
d = json.load(open('/kaggle/working/vram_opt13b.json'))['vram_scaling']
for b, r in d['by_batch'].items():
    print(f"B={b:>3}  apia={r['apia']['peak']:>7.0f}MB  lora={r['lora']['peak']:>7.0f}MB")